# 04 - Multi-Object Tracking

Tracking con salida tabular, calculo de trayectorias y ploteo directo. Las celdas de inferencia estan comentadas para que puedas activar un video real.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "vision").exists():
    PROJECT_ROOT = cwd
elif len(cwd.parents) >= 3 and (cwd.parents[2] / "vision").exists():
    PROJECT_ROOT = cwd.parents[2]
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_DIR = PROJECT_ROOT / "vision" / "yolo" / "notebooks"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE = (
    str(NOTEBOOK_DIR / "bus.jpg")
    if (NOTEBOOK_DIR / "bus.jpg").exists()
    else "https://ultralytics.com/images/bus.jpg"
)

from vision.yolo.track import (
    track_video,
    build_tracks_dataframe,
    compute_track_statistics,
    smooth_tracks,
    filter_short_tracks,
)
from vision.yolo.plotting import plot_tracking_trajectories, plot_video_statistics

MODEL = (
    str(NOTEBOOK_DIR / "yolo11n.pt")
    if (NOTEBOOK_DIR / "yolo11n.pt").exists()
    else "yolo11n.pt"
)


In [ ]:
# Ejecutar tracking sobre un video real
# VIDEO = "path/to/video.mp4"
# raw = track_video(MODEL, VIDEO, tracker="bytetrack.yaml", confidence=0.25, save_to=OUTPUT_DIR / "04_raw_tracks.parquet")
# print(raw.head())


In [ ]:
# Ejemplo minimo sin modelo para probar el analisis
import pandas as pd
raw = pd.DataFrame({
    "frame": [0, 1, 2, 0, 1, 2],
    "track_id": [1, 1, 1, 2, 2, 2],
    "class_name": ["person"] * 6,
    "confidence": [0.9, 0.88, 0.86, 0.8, 0.82, 0.84],
    "xmin": [10, 20, 30, 100, 95, 90],
    "ymin": [20, 25, 30, 60, 65, 70],
    "xmax": [60, 70, 80, 150, 145, 140],
    "ymax": [120, 125, 130, 160, 165, 170],
    "timestamp": [0.0, 0.03, 0.06, 0.0, 0.03, 0.06],
})
tracks = build_tracks_dataframe(raw)
print(tracks.head())


In [ ]:
stats = compute_track_statistics(tracks)
print(stats)
long_tracks = filter_short_tracks(tracks, min_frames=3)
smoothed = smooth_tracks(long_tracks, window=2)


In [ ]:
plot_tracking_trajectories(
    smoothed,
    image_width=200,
    image_height=200,
    save_to=OUTPUT_DIR / "04_tracking_trajectories.png",
)


In [ ]:
plot_video_statistics(raw, save_to=OUTPUT_DIR / "04_tracking_video_stats.png")
